In [3]:
# ================================
# BANK MARKETING DASHBOARD
# Term Deposit Subscription Prediction
# ================================

# Install Libraries (Run Once)
# !pip install pandas numpy matplotlib seaborn scikit-learn plotly dash shap lime

# ================================
# IMPORT LIBRARIES
# ================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    roc_curve,
    auc
)

import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc

import shap

# ================================
# LOAD DATASET
# ================================

df = pd.read_csv('bank marketing full.csv', sep=';')

print(df.head())

# ================================
# ENCODE CATEGORICAL FEATURES
# ================================

label_encoders = {}

for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le

# ================================
# FEATURES & TARGET
# ================================

X = df.drop('y', axis=1)
y = df['y']

# ================================
# TRAIN TEST SPLIT
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ================================
# SCALE DATA FOR LOGISTIC REGRESSION
# ================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ================================
# LOGISTIC REGRESSION MODEL
# ================================

log_model = LogisticRegression(
    max_iter=10000,
    solver='lbfgs'
)

log_model.fit(X_train_scaled, y_train)

# ================================
# RANDOM FOREST MODEL
# ================================

rf_model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)

rf_model.fit(X_train, y_train)

# ================================
# PREDICTIONS
# ================================

# Logistic Regression
log_pred = log_model.predict(X_test_scaled)
log_prob = log_model.predict_proba(X_test_scaled)[:,1]

# Random Forest
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

# ================================
# EVALUATION METRICS
# ================================

log_f1 = f1_score(y_test, log_pred)
rf_f1 = f1_score(y_test, rf_pred)

print("\nLogistic Regression F1 Score:", log_f1)
print("Random Forest F1 Score:", rf_f1)

# ================================
# CONFUSION MATRIX
# ================================

cm = confusion_matrix(y_test, rf_pred)

cm_fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Random Forest Confusion Matrix'
)

# ================================
# ROC CURVE
# ================================

log_fpr, log_tpr, _ = roc_curve(y_test, log_prob)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)

log_auc = auc(log_fpr, log_tpr)
rf_auc = auc(rf_fpr, rf_tpr)

roc_fig = go.Figure()

roc_fig.add_trace(
    go.Scatter(
        x=log_fpr,
        y=log_tpr,
        mode='lines',
        name=f'Logistic Regression AUC = {log_auc:.2f}'
    )
)

roc_fig.add_trace(
    go.Scatter(
        x=rf_fpr,
        y=rf_tpr,
        mode='lines',
        name=f'Random Forest AUC = {rf_auc:.2f}'
    )
)

roc_fig.add_trace(
    go.Scatter(
        x=[0,1],
        y=[0,1],
        mode='lines',
        line=dict(dash='dash'),
        name='Random Guess'
    )
)

roc_fig.update_layout(
    title='ROC Curve Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    template='plotly_dark'
)

# ================================
# FEATURE IMPORTANCE
# ================================

importance = rf_model.feature_importances_

feature_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance
})

feature_df = feature_df.sort_values(
    by='Importance',
    ascending=False
)

importance_fig = px.bar(
    feature_df.head(10),
    x='Importance',
    y='Feature',
    orientation='h',
    title='Top 10 Important Features',
    color='Importance',
    template='plotly_dark'
)

# ================================
# SHAP EXPLAINABILITY 
# ================================

import shap

# Create SHAP Explainer
explainer = shap.TreeExplainer(rf_model)

# Get SHAP values
shap_values = explainer.shap_values(X_test)

# For binary classification Random Forest
# Use class 1 values only

if isinstance(shap_values, list):

    shap.summary_plot(
        shap_values[1],
        X_test,
        show=False
    )

else:

    # Handle newer SHAP versions
    shap.summary_plot(
        shap_values[:, :, 1],
        X_test,
        show=False
    )

plt.tight_layout()

plt.savefig("shap_summary.png")

plt.close()
# ================================
# CREATE DASHBOARD
# ================================

app = Dash(__name__)

app.layout = html.Div([

    # TITLE
    html.Div([

        html.H1(
            "AI-Powered Bank Marketing Dashboard",
            style={
                'textAlign':'center',
                'color':'white'
            }
        )

    ], style={
        'backgroundColor':'#1C2541',
        'padding':'20px',
        'borderRadius':'15px',
        'marginBottom':'30px'
    }),

    # KPI CARDS
    html.Div([

        html.Div([

            html.H3(
                "Logistic Regression F1",
                style={'color':'white'}
            ),

            html.H1(
                f"{log_f1:.2f}",
                style={'color':'#00FFAA'}
            )

        ], style={
            'backgroundColor':'#1C2541',
            'padding':'20px',
            'borderRadius':'15px',
            'width':'30%',
            'textAlign':'center'
        }),

        html.Div([

            html.H3(
                "Random Forest F1",
                style={'color':'white'}
            ),

            html.H1(
                f"{rf_f1:.2f}",
                style={'color':'#00FFAA'}
            )

        ], style={
            'backgroundColor':'#1C2541',
            'padding':'20px',
            'borderRadius':'15px',
            'width':'30%',
            'textAlign':'center'
        }),

        html.Div([

            html.H3(
                "Dataset Size",
                style={'color':'white'}
            ),

            html.H1(
                f"{len(df)}",
                style={'color':'#00FFAA'}
            )

        ], style={
            'backgroundColor':'#1C2541',
            'padding':'20px',
            'borderRadius':'15px',
            'width':'30%',
            'textAlign':'center'
        })

    ], style={
        'display':'flex',
        'justifyContent':'space-between',
        'marginBottom':'30px'
    }),

    # CHARTS ROW
    html.Div([

        html.Div([
            dcc.Graph(figure=roc_fig)
        ], style={
            'width':'49%',
            'backgroundColor':'#1C2541',
            'padding':'10px',
            'borderRadius':'15px'
        }),

        html.Div([
            dcc.Graph(figure=cm_fig)
        ], style={
            'width':'49%',
            'backgroundColor':'#1C2541',
            'padding':'10px',
            'borderRadius':'15px'
        })

    ], style={
        'display':'flex',
        'justifyContent':'space-between',
        'marginBottom':'30px'
    }),

    # FEATURE IMPORTANCE
    html.Div([

        dcc.Graph(figure=importance_fig)

    ], style={
        'backgroundColor':'#1C2541',
        'padding':'20px',
        'borderRadius':'15px'
    })

], style={
    'backgroundColor':'#0D1B2A',
    'padding':'30px',
    'fontFamily':'Arial'
})

# ================================
# RUN DASHBOARD
# ================================

app.run(debug=True)

   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  

Logistic Regression F1 Score: 0.3185483870967742
Random Forest F1 Score: 0.4972067039106145
